# NLP Disaster Tweets: Label Denoising & Multi-Model Stacking (Top Tier Solution)
**Author**: davidzwang  
**Competition**: [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/competitions/nlp-getting-started)  
**Evaluation Metric**: F1-Score  
**Best Public Score**: **0.84431** 🚀

---
## 1. Problem Formulation & Methodology Overview
Classifying Twitter short messages to distinguish actual real-world disasters (`target == 1`) from metaphorical, hyperbolic, or figurative uses of disaster terminology (`target == 0`, e.g., "this song is fire!").

### Key Engineering & Modeling Highlights:
1. **Context-Preserving Text Normalization**:
   - Decoding HTML entities (`&amp;`, `&lt;`).
   - Normalizing URLs to a dedicated `[URL]` token and user handles to `[USER]` token while preserving punctuation exclamation/question marks indicative of urgency.
2. **Train Set Ground Truth Label Denoising**:
   - Identifying identical tweet texts in the training dataset that had contradictory labels due to annotation noise; resolving ambiguities using exact text group majority aggregation.
3. **Dual Word & Character N-Gram Feature Engineering**:
   - Word n-grams ($1-3$) with sublinear TF scaling to capture phrases and terminology.
   - Character n-grams ($3-5$) to capture typos, informal slang, hashtags, and subword stems.
   - Meta-linguistic statistics: Capitalization ratio, exclamation frequency, punctuation density, average token length.
4. **5-Fold Stratified Stacking Ensemble**:
   - Combining distinct linear and probabilistic classification engines: **Logistic Regression** (L2-regularized), **Ridge Classifier**, **Complement Naive Bayes** (adapted for imbalanced text collections), and **SGD Classifier** (modified Huber loss).
5. **Exact Match Duplication Post-Processing**:
   - For test tweets possessing exact identical matches in the curated train set, directly overriding probabilistic predictions with the consensus train label.


In [ ]:
import os
import re
import html
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

# Detect environment
if os.path.exists('/kaggle/input/nlp-getting-started'):
    DATA_DIR = '/kaggle/input/nlp-getting-started'
else:
    DATA_DIR = 'data'

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f'Train shape: {train.shape}, Test shape: {test.shape}')
train.head()


## 2. Text Normalization, Denoising & Meta-Features

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)
    text = re.sub(r"@\w+", " [USER] ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def denoise_train_labels(df):
    df = df.copy()
    text_stats = df.groupby('text')['target'].agg(['count', 'mean'])
    conflicts = text_stats[(text_stats['count'] > 1) & (text_stats['mean'] > 0) & (text_stats['mean'] < 1)]
    for text, row in conflicts.iterrows():
        majority = 1 if row['mean'] >= 0.5 else 0
        df.loc[df['text'] == text, 'target'] = majority
    print(f'Denoised {len(conflicts)} conflicting text groups using majority consensus.')
    return df

def extract_meta_features(df):
    feats = pd.DataFrame(index=df.index)
    text_series = df['text'].fillna("")
    feats['char_len'] = text_series.apply(len)
    feats['word_count'] = text_series.apply(lambda x: len(x.split()))
    feats['mean_word_len'] = feats['char_len'] / (feats['word_count'] + 1e-5)
    feats['caps_count'] = text_series.apply(lambda x: sum(1 for c in x if c.isupper()))
    feats['caps_ratio'] = feats['caps_count'] / (feats['char_len'] + 1e-5)
    feats['excl_count'] = text_series.apply(lambda x: x.count('!'))
    feats['quest_count'] = text_series.apply(lambda x: x.count('?'))
    feats['has_keyword'] = df['keyword'].notnull().astype(int)
    return feats

train = denoise_train_labels(train)
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

train_meta = extract_meta_features(train)
test_meta = extract_meta_features(test)


## 3. Dual N-Gram Vectorization & 5-Fold Stratified Stacking

In [ ]:
# Dual Vectorizers: Word & Char
word_vec = TfidfVectorizer(ngram_range=(1, 3), max_features=12000, sublinear_tf=True)
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer='char', max_features=15000, sublinear_tf=True)

all_text = pd.concat([train['clean_text'], test['clean_text']])
word_vec.fit(all_text)
char_vec.fit(all_text)

X_train_word = word_vec.transform(train['clean_text'])
X_test_word = word_vec.transform(test['clean_text'])

X_train_char = char_vec.transform(train['clean_text'])
X_test_char = char_vec.transform(test['clean_text'])

X_train = hstack([X_train_word, X_train_char, train_meta.values]).tocsr()
X_test = hstack([X_test_word, X_test_char, test_meta.values]).tocsr()
y_train = train['target'].values
print(f'Total feature dimensions: {X_train.shape[1]}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': LogisticRegression(C=1.2, max_iter=1000),
    'RidgeClassifier': RidgeClassifier(alpha=1.0),
    'SGDClassifier': SGDClassifier(loss='modified_huber', alpha=1e-4, max_iter=1000, random_state=42),
    'ComplementNB': ComplementNB(alpha=0.8)
}

oof_preds = {name: np.zeros(len(y_train)) for name in models}
test_preds = {name: np.zeros(X_test.shape[0]) for name in models}

for name, model in models.items():
    print(f'Training 5-Fold {name}...')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
        X_va, y_va = X_train[val_idx], y_train[val_idx]
        
        model.fit(X_tr, y_tr)
        if hasattr(model, 'predict_proba'):
            oof_preds[name][val_idx] = model.predict_proba(X_va)[:, 1]
            test_preds[name] += model.predict_proba(X_test)[:, 1] / 5
        elif hasattr(model, 'decision_function'):
            df_va = model.decision_function(X_va)
            df_te = model.decision_function(X_test)
            oof_preds[name][val_idx] = 1 / (1 + np.exp(-df_va))
            test_preds[name] += (1 / (1 + np.exp(-df_te))) / 5

oof_ensemble = sum(oof_preds.values()) / len(models)
test_ensemble = sum(test_preds.values()) / len(models)

# Optimal Threshold Search
best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.30, 0.70, 0.01):
    score = f1_score(y_train, (oof_ensemble > th).astype(int))
    if score > best_f1:
        best_f1, best_th = score, th

print(f'Optimal Decision Threshold: {best_th:.2f}, 5-Fold OOF F1-Score: {best_f1:.4f}')


## 4. Exact Text Match Duplication Post-Processing & Export

In [ ]:
final_test_preds = (test_ensemble > best_th).astype(int)

# Exact text match post-processing
train_lookup = train.groupby('text')['target'].agg(lambda x: x.mode()[0]).to_dict()
override_count = 0
for idx, text in enumerate(test['text']):
    if text in train_lookup:
        final_test_preds[idx] = int(train_lookup[text])
        override_count += 1

print(f'Overrode {override_count} predictions with exact match ground truth labels.')

sub = pd.DataFrame({
    'id': test['id'],
    'target': final_test_preds
})
sub.to_csv('submission.csv', index=False)
print('Saved final submission to submission.csv.')
print(sub['target'].value_counts(normalize=True))
